<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/external_dataset_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Checking the external datasets

Our supervisor told us that the second dataset does not have to be IEEE-CIS. Our final aim is a prototype that takes a transaction dataset and predicts fraud, so we want to validate our method on two more datasets before we build it. We are considering:

1. **Credit Card Fraud Detection Dataset 2026** (`mohankrishnathalla/credit-card-fraud-detection-dataset-2026`): 100,000 transactions in 2026, 2.06% fraud, with card ID, full timestamp, amount, merchant category, age and gender.
2. **Financial Transactions Dataset for Fraud Detection** (`aryan208/financial-transactions-dataset-for-fraud-detection`): a large synthetic payments dataset. We have not checked its columns yet.

Before we spend time on full experiments, we check each dataset in this notebook:

- **Overview:** rows, columns, fraud rate and time range.
- **Column mapping:** which columns we can use for our six features (`amt_log`, `category_risk`, `is_night`, `gap_log`, `age`, `gender_flag`).
- **Cards and time gaps:** how many transactions each card has, whether fraud comes in bursts like in Sparkov, and whether fraud transactions have shorter time gaps.
- **Each column on its own:** the test PR-AUC and ROC-AUC of every raw column, to find columns that leak the label (scores close to 1) and to see whether the labels carry any signal at all.
- **Quick model check:** one Random Forest on our features, with and without `gap_log`.

This is only a check, so we use one seed and a simple 80/20 split by time, with no bootstrap. The full validation with our protocol comes after we choose the datasets.

## 1. Environment setup
We type credentials in at runtime, so they are never saved in the notebook.

In [1]:
# 1.1 Download both datasets from Kaggle
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')

DATASETS = {
    'cc2026':  'mohankrishnathalla/credit-card-fraud-detection-dataset-2026',
    'fintx':   'aryan208/financial-transactions-dataset-for-fraud-detection',
}
for short, slug in DATASETS.items():
    !kaggle datasets download -d {slug} -p /content/data/external/{short} --unzip

Kaggle username: Subbareddy_subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/mohankrishnathalla/credit-card-fraud-detection-dataset-2026
License(s): CC0-1.0
100% 6.65M/6.65M [00:01<00:00, 4.46MB/s]

Dataset URL: https://www.kaggle.com/datasets/aryan208/financial-transactions-dataset-for-fraud-detection
License(s): CC0-1.0
100% 277M/277M [00:16<00:00, 17.8MB/s]



In [2]:
# 1.2 Clone the project repository (to use our helpers and save the results)
github_token = getpass('Paste your GitHub token: ').strip()
repo_url = f'https://{github_token}@github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git'
!git clone {repo_url}
%cd Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 257 (delta 79), reused 42 (delta 42), pack-reused 149 (from 1)
Receiving objects: 100% (257/257), 11.60 MiB | 14.39 MiB/s, done.
Resolving deltas: 100% (157/157), done.
/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [3]:
# 1.3 Imports and project helpers
import sys, glob
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

sys.path.append(os.getcwd())
from src.utils import fit_category_risk, apply_category_risk
pd.set_option('display.max_columns', 60)

## 2. Loading the data
Each download may contain more than one file, so we list them first and load the largest CSV.

In [4]:
# 2.1 List the downloaded files
for short in DATASETS:
    print(f'--- {short}')
    for f in sorted(glob.glob(f'/content/data/external/{short}/**/*', recursive=True)):
        if os.path.isfile(f):
            print(f'{os.path.getsize(f) / 1e6:9.1f} MB  {f}')

--- cc2026
     25.7 MB  /content/data/external/cc2026/credit_card_fraud_detection_2026.csv
--- fintx
    796.1 MB  /content/data/external/fintx/financial_fraud_detection_dataset.csv


In [5]:
# 2.2 Load the largest CSV of each dataset
raw = {}
for short in DATASETS:
    csvs = glob.glob(f'/content/data/external/{short}/**/*.csv', recursive=True)
    main_csv = max(csvs, key=os.path.getsize)
    raw[short] = pd.read_csv(main_csv, low_memory=False)
    print(f'{short}: {main_csv}  ->  {raw[short].shape[0]:,} rows, {raw[short].shape[1]} columns')

cc2026: /content/data/external/cc2026/credit_card_fraud_detection_2026.csv  ->  100,000 rows, 43 columns
fintx: /content/data/external/fintx/financial_fraud_detection_dataset.csv  ->  5,000,000 rows, 18 columns


## 3. Overview
For every column we show its type, the number of distinct values, the share of missing values and an example value.

In [6]:
# 3.1 Column overview of each dataset
def column_overview(df):
    return pd.DataFrame({
        'type': df.dtypes.astype(str),
        'distinct values': df.nunique(),
        'missing %': (df.isna().mean() * 100).round(2),
        'example': df.iloc[0],
    })

for short, df in raw.items():
    print(f'=== {short}')
    display(column_overview(df))
    display(df.head(3))

=== cc2026


,type,distinct values,missing %,example
transaction_id,object,100000,0.00,TXN-00000001
customer_id,object,8000,0.00,CUS-00006042
card_id,object,8000,0.00,CRD-00006042
merchant_id,object,3000,0.00,MER-00002507
transaction_datetime,object,99762,0.00,2026-01-01 02:28:51
transaction_amount,float64,30862,0.00,37.36
merchant_category,object,15,0.00,Healthcare & Pharmacy
merchant_country,object,20,0.00,USA
transaction_hour,int64,24,0.00,2
transaction_day_of_week,object,7,0.00,Monday


,transaction_id,customer_id,card_id,merchant_id,transaction_datetime,transaction_amount,merchant_category,merchant_country,transaction_hour,transaction_day_of_week,card_present,customer_age,customer_gender,customer_state,account_age_days,credit_limit,credit_score,num_cards_held,transactions_last_1h,transactions_last_24h,transactions_last_7d,amount_last_1h,amount_last_24h,days_since_last_transaction,distance_from_home_km,distance_from_last_tx_km,international_transaction,high_risk_country,amount_vs_avg_ratio,used_chip,used_pin,transaction_channel,device_type,browser,is_weekend,avg_transaction_amount_30d,std_transaction_amount_30d,num_declined_last_30d,num_chargebacks_ever,email_domain,is_fraud,fraud_type,fraud_amount_lost
0,TXN-00000001,CUS-00006042,CRD-00006042,MER-00002507,2026-01-01 02:28:51,37.36,Healthcare & Pharmacy,USA,2,Monday,0,41,Female,PA,709,6630.0,742,2,1,2,3,88.91,209.94,0.9,13.0,0.3,0,0,0.398,0,0,Online,Mobile,Chrome,0,93.91,39.48,0,0,corporate,0,Legitimate,0.0
1,TXN-00000002,CUS-00005262,CRD-00005262,MER-00000189,2026-01-01 03:15:18,50.04,Restaurant & Dining,Germany,3,Monday,1,56,Male,WV,1143,4350.0,602,1,6,6,23,1290.10,1093.31,6.6,5311.8,17.0,1,0,0.278,1,1,In-store,POS Terminal,NaN,0,180.05,134.45,0,0,yahoo,0,Legitimate,0.0
2,TXN-00000003,CUS-00005901,CRD-00005901,MER-00002054,2026-01-01 03:44:38,38.87,Grocery & Supermarket,Australia,3,Monday,0,43,Male,MN,601,3950.0,605,4,1,1,3,36.27,46.54,4.2,14164.5,53.2,1,0,0.925,0,0,Online,Desktop,Chrome,0,42.00,22.46,0,0,hotmail,0,Legitimate,0.0


=== fintx


,type,distinct values,missing %,example
transaction_id,object,5000000,0.00,T100000
timestamp,object,4999998,0.00,2023-08-22T09:22:43.516168
sender_account,object,896513,0.00,ACC877572
receiver_account,object,896639,0.00,ACC388389
amount,float64,217069,0.00,343.78
transaction_type,object,4,0.00,withdrawal
merchant_category,object,8,0.00,utilities
location,object,8,0.00,Tokyo
device_used,object,4,0.00,mobile
is_fraud,bool,2,0.00,False


,transaction_id,timestamp,sender_account,receiver_account,amount,transaction_type,merchant_category,location,device_used,is_fraud,fraud_type,time_since_last_transaction,spending_deviation_score,velocity_score,geo_anomaly_score,payment_channel,ip_address,device_hash
0,T100000,2023-08-22T09:22:43.516168,ACC877572,ACC388389,343.78,withdrawal,utilities,Tokyo,mobile,False,NaN,NaN,-0.21,3,0.22,card,13.101.214.112,D8536477
1,T100001,2023-08-04T01:58:02.606711,ACC895667,ACC944962,419.65,withdrawal,online,Toronto,atm,False,NaN,NaN,-0.14,7,0.96,ACH,172.52.47.194,D2622631
2,T100002,2023-05-12T11:39:33.742963,ACC733052,ACC377370,2773.86,deposit,other,London,pos,False,NaN,NaN,-1.78,20,0.89,card,185.98.35.23,D4823498


## 4. Column mapping
Here we tell the notebook which column plays which role. Any role left as `None` is filled in automatically from a list of common names. We print the result, so we can correct a wrong guess by editing `MAPPINGS` and running the cell again.

`EXCLUDE` lists columns we never use as features: other target columns (such as `fraud_type` or `fraud_amount_lost`) describe the fraud itself, so using them would leak the label.

In [7]:
# 4.1 Roles of the columns in each dataset
MAPPINGS = {
    'cc2026': {'label': 'is_fraud', 'id': 'card_id', 'time': 'transaction_datetime', 'amount': 'amount',
               'category': 'merchant_category', 'age': 'age', 'gender': 'gender'},
    'fintx':  {'label': None, 'id': None, 'time': None, 'amount': None,
               'category': None, 'age': None, 'gender': None},
}
EXCLUDE = {
    'cc2026': ['fraud_type', 'fraud_amount_lost'],
    'fintx':  ['fraud_type'],
}

CANDIDATES = {
    'label':    ['is_fraud', 'isFraud', 'fraud', 'is_fraudulent', 'label', 'Class'],
    'id':       ['card_id', 'cc_num', 'customer_id', 'sender_account', 'account_id', 'nameOrig', 'user_id', 'customer'],
    'time':     ['transaction_datetime', 'trans_date_trans_time', 'timestamp', 'datetime', 'transaction_time', 'date', 'step'],
    'amount':   ['amount', 'amt', 'transaction_amount'],
    'category': ['merchant_category', 'category', 'transaction_type', 'type'],
    'age':      ['age', 'customer_age'],
    'gender':   ['gender', 'sex'],
}

def fill_mapping(df, mapping):
    out = dict(mapping)
    lower = {c.lower(): c for c in df.columns}
    for role, col in mapping.items():
        if col is None:
            out[role] = next((lower[c.lower()] for c in CANDIDATES[role] if c.lower() in lower), None)
        elif col not in df.columns:
            print(f'warning: column {col!r} for {role} not found')
            out[role] = None
    return out

for short, df in raw.items():
    MAPPINGS[short] = fill_mapping(df, MAPPINGS[short])
    EXCLUDE[short] = [c for c in EXCLUDE[short] if c in df.columns]
    print(short, MAPPINGS[short], '| excluded:', EXCLUDE[short])

cc2026 {'label': 'is_fraud', 'id': 'card_id', 'time': 'transaction_datetime', 'amount': None, 'category': 'merchant_category', 'age': None, 'gender': None} | excluded: ['fraud_type', 'fraud_amount_lost']
fintx {'label': 'is_fraud', 'id': 'sender_account', 'time': 'timestamp', 'amount': 'amount', 'category': 'merchant_category', 'age': None, 'gender': None} | excluded: ['fraud_type']


In [8]:
# 4.2 Parse the label and the time column
def parse_time(s):
    # a datetime column gives us hours; a numeric step column (as in PaySim) is kept as it is
    if pd.api.types.is_numeric_dtype(s):
        return s, False
    return pd.to_datetime(s, errors='coerce'), True

data, has_clock = {}, {}
for short, df in raw.items():
    m = MAPPINGS[short]
    d = df.copy()
    d['_y'] = pd.to_numeric(d[m['label']], errors='coerce').fillna(0).astype(int)
    d['_time'], has_clock[short] = parse_time(d[m['time']])
    print(f"{short}: time parsed as datetime = {has_clock[short]},  unparsed times = {d['_time'].isna().sum():,}")
    data[short] = d.dropna(subset=['_time']).sort_values('_time').reset_index(drop=True)

cc2026: time parsed as datetime = True,  unparsed times = 0
fintx: time parsed as datetime = True,  unparsed times = 3


## 5. Fraud rate, time range and cards
For each dataset we report the fraud rate and time range, how many transactions each card has, and how fraud is spread over cards and days. In Sparkov, fraud comes in bursts of several transactions on the same card on one day, and fraud transactions have much shorter time gaps (median 1.33 h against 4.57 h). If a dataset shows the same pattern, we expect `gap_log` to help there too.

In [9]:
# 5.1 Basic facts and the time gaps of each dataset
facts = []
for short, d in data.items():
    m = MAPPINGS[short]
    row = {'dataset': short, 'rows': len(d), 'columns': raw[short].shape[1],
           'fraud rows': int(d['_y'].sum()), 'fraud rate %': round(d['_y'].mean() * 100, 3),
           'start': d['_time'].min(), 'end': d['_time'].max()}
    if m['id'] is not None:
        per_card = d.groupby(m['id']).size()
        gap = d.sort_values([m['id'], '_time']).groupby(m['id'])['_time'].diff()
        gap_h = gap.dt.total_seconds() / 3600 if has_clock[short] else gap
        d['_gap_h'] = gap_h.reindex(d.index)
        row.update({'cards': per_card.size,
                    'median txns per card': per_card.median(),
                    'cards with 1 txn %': round((per_card == 1).mean() * 100, 2),
                    'median gap, fraud': d.loc[d['_y'] == 1, '_gap_h'].median(),
                    'median gap, normal': d.loc[d['_y'] == 0, '_gap_h'].median()})
        if has_clock[short]:
            f = d[d['_y'] == 1]
            per_card_day = f.groupby([m['id'], f['_time'].dt.floor('D')]).size()
            row.update({'mean frauds per card-day': round(per_card_day.mean(), 2),
                        'frauds in card-days with 2+ frauds %': round(per_card_day[per_card_day >= 2].sum() / per_card_day.sum() * 100, 1),
                        'fraud at night (22-03) %': round(f['_time'].dt.hour.isin([22, 23, 0, 1, 2, 3]).mean() * 100, 1),
                        'normal at night (22-03) %': round(d.loc[d['_y'] == 0, '_time'].dt.hour.isin([22, 23, 0, 1, 2, 3]).mean() * 100, 1)})
    facts.append(row)

facts_df = pd.DataFrame(facts).set_index('dataset').T
facts_df

dataset,cc2026,fintx
rows,100000,4999997
columns,43,18
fraud rows,2055,179553
fraud rate %,2.055,3.591
start,2026-01-01 02:28:51,2023-01-01 00:09:26.241974
end,2026-12-31 23:45:47,2024-01-01 22:58:30.131850
cards,8000,896513
median txns per card,12.0,5.0
cards with 1 txn %,0.01,2.14
"median gap, fraud",444.14,860.340349


## 6. Each column on its own
We split each dataset by time (first 80% of the period for fitting, last 20% for testing) and score every raw column on its own on the test part:
- a **numeric column** is used directly as a score. We decide on the fitting part whether high or low values mean fraud;
- a **text column** with up to 1,000 values is replaced by its fraud rate on the fitting part;
- **identifiers** (IDs, account numbers, hashes, IP addresses) and columns with more than 1,000 text values are skipped.

How we read the results:
- **ROC-AUC of 0.95 or more:** the column almost gives the answer, so we check it for leakage before using it;
- **ROC-AUC below 0.55:** the column has little or no signal. If every column is like this, the labels may have been assigned at random and the dataset cannot validate anything;
- **lift** is the PR-AUC divided by the fraud rate of the test part. A lift of 1 means no better than guessing.

In [10]:
# 6.1 Split each dataset by time
def time_split(d, frac=0.8):
    cutoff = d['_time'].min() + frac * (d['_time'].max() - d['_time'].min())
    return d[d['_time'] < cutoff].copy(), d[d['_time'] >= cutoff].copy(), cutoff

splits = {}
for short, d in data.items():
    fit_part, test_part, cutoff = time_split(d)
    splits[short] = (fit_part, test_part)
    print(f"{short}: cutoff {cutoff}  fit {len(fit_part):,} rows ({fit_part['_y'].mean():.3%} fraud)  "
          f"test {len(test_part):,} rows ({test_part['_y'].mean():.3%} fraud)")

cc2026: cutoff 2026-10-20 00:18:23.800000  fit 79,982 rows (2.047% fraud)  test 20,018 rows (2.088% fraud)
fintx: cutoff 2023-10-20 18:24:41.353874800  fit 4,004,296 rows (3.588% fraud)  test 995,701 rows (3.603% fraud)


In [11]:
# 6.2 Score every raw column on its own
def looks_like_identifier(col, mapping):
    c = col.lower()
    return (col == mapping['id'] or c in ('id', 'ip_address') or c.endswith('_id')
            or 'account' in c or 'hash' in c or c.endswith('_ip'))

def column_scores(short):
    m = MAPPINGS[short]
    fit_part, test_part = splits[short]
    y_fit, y_test = fit_part['_y'], test_part['_y']
    rows = []
    for col in raw[short].columns:
        if col == m['label'] or col in EXCLUDE[short]:
            continue
        s_fit, s_test = fit_part[col], test_part[col]
        if col == m['time']:
            rows.append({'column': col, 'used as': 'skipped (time column)'}); continue
        if looks_like_identifier(col, m):
            rows.append({'column': col, 'used as': 'skipped (identifier)'}); continue
        if pd.api.types.is_numeric_dtype(s_fit):
            fill = s_fit.astype(float).median()
            x_fit = s_fit.astype(float).fillna(fill)
            x_test = s_test.astype(float).fillna(fill)
            if x_fit.nunique() > 1 and roc_auc_score(y_fit, x_fit) < 0.5:
                x_test = -x_test                      # low values mean fraud
            used = 'numeric'
        elif s_fit.nunique() <= 1000:
            rates = y_fit.groupby(s_fit.astype(str)).mean()
            x_test = s_test.astype(str).map(rates).fillna(y_fit.mean())
            used = 'fraud rate of each value'
        else:
            rows.append({'column': col, 'used as': 'skipped (too many text values)'}); continue
        if x_test.nunique() < 2:
            rows.append({'column': col, 'used as': used + ' (constant on test)'}); continue
        pr, roc = average_precision_score(y_test, x_test), roc_auc_score(y_test, x_test)
        rows.append({'column': col, 'used as': used, 'PR-AUC': round(pr, 4),
                     'lift': round(pr / y_test.mean(), 2), 'ROC-AUC': round(roc, 4)})
    out = pd.DataFrame(rows)
    out['note'] = np.select([out['ROC-AUC'] >= 0.95, out['ROC-AUC'] < 0.55],
                            ['check for leakage', 'little or no signal'], 'some signal')
    out.loc[out['ROC-AUC'].isna(), 'note'] = ''
    return out.sort_values('ROC-AUC', ascending=False, na_position='last').reset_index(drop=True)

column_results = {}
for short in data:
    column_results[short] = column_scores(short)
    print(f'=== {short}  (test fraud rate {splits[short][1]["_y"].mean():.3%})')
    display(column_results[short])

=== cc2026  (test fraud rate 2.088%)


,column,used as,PR-AUC,lift,ROC-AUC,note
0,merchant_country,fraud rate of each value,0.0565,2.71,0.7344,some signal
1,international_transaction,numeric,0.0336,1.61,0.6873,some signal
2,distance_from_home_km,numeric,0.0316,1.51,0.6716,some signal
3,browser,fraud rate of each value,0.0296,1.42,0.6417,some signal
4,transaction_channel,fraud rate of each value,0.0297,1.42,0.6402,some signal
5,device_type,fraud rate of each value,0.0283,1.35,0.6263,some signal
6,card_present,numeric,0.0272,1.30,0.6147,some signal
7,used_chip,numeric,0.0267,1.28,0.6089,some signal
8,high_risk_country,numeric,0.0421,2.01,0.6020,some signal
9,transactions_last_1h,numeric,0.0332,1.59,0.5879,some signal


=== fintx  (test fraud rate 3.603%)


,column,used as,PR-AUC,lift,ROC-AUC,note
0,time_since_last_transaction,numeric,0.0435,1.21,0.5860,some signal
1,velocity_score,numeric,0.0363,1.01,0.5021,little or no signal
2,geo_anomaly_score,numeric,0.0361,1.00,0.5018,little or no signal
3,device_used,fraud rate of each value,0.0362,1.00,0.5012,little or no signal
4,transaction_type,fraud rate of each value,0.0362,1.00,0.5005,little or no signal
5,amount,numeric,0.0361,1.00,0.5002,little or no signal
6,payment_channel,fraud rate of each value,0.0360,1.00,0.4997,little or no signal
7,spending_deviation_score,numeric,0.0360,1.00,0.4991,little or no signal
8,merchant_category,fraud rate of each value,0.0359,1.00,0.4983,little or no signal
9,location,fraud rate of each value,0.0358,0.99,0.4978,little or no signal


## 7. Quick model check
We build our features from the raw columns only, in the same way as on Sparkov, and ignore any ready-made features in the dataset:
- `amt_log`: log of the amount;
- `category_risk`: the fraud rate of each category on the fitting part;
- `is_night`: 1 for transactions between 22:00 and 03:59 (only when the time column has hours);
- `gap_log`: log of the hours since the card's previous transaction, -1 for a card's first transaction;
- `age_num` (the age as a number, -1 when missing) and `gender_flag`, when the dataset has them.

We train one Random Forest with the Random Forest Tuned settings and seed 42, once with all available features and once without `gap_log`, and compare the test PR-AUC. Datasets with more than 1.5 million rows are reduced to a random sample of cards, keeping every transaction of each chosen card, so the time gaps stay correct.

In [12]:
# 7.1 Build our features from the raw columns
NIGHT_HOURS = [22, 23, 0, 1, 2, 3]
MAX_ROWS = 1_500_000

def build_features(short):
    m, d = MAPPINGS[short], data[short].copy()
    features = []
    if m['amount']:
        d['amt_log'] = np.log1p(pd.to_numeric(d[m['amount']], errors='coerce').clip(lower=0)).fillna(0)
        features.append('amt_log')
    if m['category']:
        features.append('category_risk')                 # filled in after the split
    if has_clock[short]:
        d['is_night'] = d['_time'].dt.hour.isin(NIGHT_HOURS).astype(int)
        features.append('is_night')
    if m['id']:
        d['gap_log'] = np.log1p(d['_gap_h']).fillna(-1)
        features.append('gap_log')
    if m['age']:
        d['age_num'] = pd.to_numeric(d[m['age']].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce').fillna(-1)
        features.append('age_num')
    if m['gender']:
        d['gender_flag'] = d[m['gender']].astype(str).str.strip().str.upper().str.startswith('M').astype(int)
        features.append('gender_flag')

    if len(d) > MAX_ROWS and m['id']:
        cards = d[m['id']].unique()
        keep = np.random.default_rng(42).choice(cards, size=int(len(cards) * MAX_ROWS / len(d)), replace=False)
        d = d[d[m['id']].isin(keep)]
        print(f'{short}: sampled {len(keep):,} cards -> {len(d):,} rows')
    return d, features

model_rows = []
for short in data:
    d, features = build_features(short)
    fit_part, test_part, _ = time_split(d)
    if 'category_risk' in features:
        cat = MAPPINGS[short]['category']
        risk = fit_category_risk(fit_part, cat, '_y')
        for part in (fit_part, test_part):
            part['category_risk'] = apply_category_risk(part, cat, risk)
    print(f'{short}: features {features}')

    # 7.2 Train with and without gap_log
    for name, cols in [('all features', features),
                       ('without gap_log', [f for f in features if f != 'gap_log'])]:
        if name == 'without gap_log' and 'gap_log' not in features:
            continue
        rf = RandomForestClassifier(n_estimators=200, max_features=min(3, len(cols)),
                                    min_samples_leaf=5, random_state=42, n_jobs=-1)
        rf.fit(fit_part[cols], fit_part['_y'])
        p = rf.predict_proba(test_part[cols])[:, 1]
        pr = average_precision_score(test_part['_y'], p)
        model_rows.append({'dataset': short, 'run': name, 'features': ', '.join(cols),
                           'test fraud rate %': round(test_part['_y'].mean() * 100, 3),
                           'PR-AUC': round(pr, 4), 'lift': round(pr / test_part['_y'].mean(), 2),
                           'ROC-AUC': round(roc_auc_score(test_part['_y'], p), 4)})
        print(f'  {name}: PR-AUC {pr:.4f}')

model_df = pd.DataFrame(model_rows)
model_df

cc2026: features ['category_risk', 'is_night', 'gap_log']
  all features: PR-AUC 0.0244
  without gap_log: PR-AUC 0.0269
fintx: sampled 268,954 cards -> 1,499,722 rows
fintx: features ['amt_log', 'category_risk', 'is_night', 'gap_log']
  all features: PR-AUC 0.0362
  without gap_log: PR-AUC 0.0359


,dataset,run,features,test fraud rate %,PR-AUC,lift,ROC-AUC
0,cc2026,all features,"category_risk, is_night, gap_log",2.088,0.0244,1.17,0.5461
1,cc2026,without gap_log,"category_risk, is_night",2.088,0.0269,1.29,0.5806
2,fintx,all features,"amt_log, category_risk, is_night, gap_log",3.585,0.0362,1.01,0.5031
3,fintx,without gap_log,"amt_log, category_risk, is_night",3.585,0.0359,1.00,0.5019


## 8. Saving the results

In [13]:
# 8.1 Save the results tables
os.makedirs('results', exist_ok=True)
facts_df.to_csv('results/external_check_facts.csv')
for short, df in column_results.items():
    df.to_csv(f'results/external_check_columns_{short}.csv', index=False)
model_df.to_csv('results/external_check_model.csv', index=False)
print('saved to results/')

saved to results/


## 9. Discussion